In [ ]:
from pathlib import Path
import json
import os
import sqlite3

from dotenv import load_dotenv
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle, resample

load_dotenv()


In [ ]:
current_dir = Path.cwd()
project_root_dir = current_dir.parent

db_path = project_root_dir / Path(
    os.getenv(
        "DATABASE_PATH"
    )
)
csv_path = project_root_dir / "water_potability.csv"
metrics_path = project_root_dir / "metrics/validation_metrics.json"
query = "SELECT * FROM water_potability"

if not db_path.exists():
    db_path.parent.mkdir(parents=True, exist_ok=True)
    df_csv = pd.read_csv(csv_path)
    with sqlite3.connect(db_path) as connection:
        df_csv.to_sql("water_potability", connection, if_exists="replace", index=False)

with sqlite3.connect(db_path) as connection:
    df = pd.read_sql_query(query, connection)

print(df.shape)
df.head()


In [ ]:
df.describe()


In [ ]:
missing_counts = df.isna().sum().sort_values(ascending=False)
duplicate_count = int(df.duplicated().sum())

missing_summary = (
    pd.DataFrame({
        "missing_count": missing_counts
    })
    .sort_values("missing_count", ascending=False)
)

missing_summary


In [ ]:
missing_plot = missing_summary.reset_index().rename(columns={"index": "feature"})

plt.figure(figsize=(10, 5))
sns.barplot(data=missing_plot, x="feature", y="missing_count")
plt.title("Braki danych")
plt.xlabel("Cecha")
plt.ylabel("Liczba braków")
plt.xticks(rotation=45)
plt.show()


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Potability")
plt.title("Rozkład klasy docelowej")
plt.xlabel("Potability")
plt.ylabel("Liczba obserwacji")
plt.tight_layout()
plt.show()


In [ ]:
df.hist(figsize=(16, 12), bins=30, edgecolor="black")
plt.suptitle("Histogramy cech numerycznych")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 8))
correlation_matrix = df.corr(numeric_only=True)
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Macierz korelacji")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.violinplot(data=df, x="Potability", y="ph", inner="quartile")
plt.title("Rozkład ph względem klasy Potability")
plt.xlabel("Potability")
plt.ylabel("ph")
plt.tight_layout()
plt.show()


In [ ]:
df.Potability.value_counts()

notpotable  = data[data['Potability']==0]
potable = data[data['Potability']==1]  

df_minority_upsampled = resample(potable, replace = True, n_samples = 1998) 

data = pd.concat([notpotable, df_minority_upsampled])
data = shuffle(data) 

In [ ]:
df_model = df.copy()

X = df_model.drop(columns=["Potability"])
y = df_model["Potability"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp,
)

In [ ]:
baseline_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                class_weight="balanced",
                n_jobs=-1,
            ),
        ),
    ]
)

baseline_model.fit(X_train, y_train)
val_pred = baseline_model.predict(X_val)

validation_metrics = {
    "accuracy": round(float(accuracy_score(y_val, val_pred)), 4),
    "precision": round(float(precision_score(y_val, val_pred)), 4),
    "recall": round(float(recall_score(y_val, val_pred)), 4),
    "f1": round(float(f1_score(y_val, val_pred)), 4)
}

pd.Series(validation_metrics, name="value").to_frame()


In [ ]:
metrics_path.parent.mkdir(parents=True, exist_ok=True)
with metrics_path.open("w", encoding="utf-8") as file:
    json.dump(validation_metrics, file, indent=2)


In [ ]:
ConfusionMatrixDisplay.from_predictions(y_val, val_pred)
plt.title("Macierz pomyłek na zbiorze walidacyjnym")
plt.tight_layout()
plt.show()
